# Tuen Mun hourly AQHI cleaning and quality checks
Read the 15 unmodified EPD English monthly CSVs in `data/raw`. Run all cells from the repository root with a Python 3 kernel and pandas installed. Output files are written under `data/processed`.

The timestamp is the Hong Kong hour endpoint (UTC+08:00): source hour 01 is 01:00 on the source date and hour 24 is midnight the following day. Daily Max rows are excluded.

In [1]:
from pathlib import Path
from datetime import date, datetime, timedelta
import csv
import re
import hashlib

import pandas as pd

raw_dir = Path("data/raw")
output_dir = Path("data/processed")
months = pd.period_range("2025-04", "2026-06", freq="M")
rows, date_hour_issues, invalid_aqhi = [], [], []
handoff_path = output_dir / "tuen_mun_aqhi.csv"
original_handoff = pd.read_csv(handoff_path) if handoff_path.exists() else None
original_handoff_hash = hashlib.sha256(handoff_path.read_bytes()).hexdigest() if handoff_path.exists() else None
raw_hashes_before = {
    f"{month.strftime('%Y%m')}_Eng.csv": hashlib.sha256(
        (raw_dir / f"{month.strftime('%Y%m')}_Eng.csv").read_bytes()
    ).hexdigest()
    for month in months
}

for month in months:
    path = raw_dir / f"{month.strftime('%Y%m')}_Eng.csv"
    current_date = None
    previous_hour = None
    with path.open(encoding="utf-8-sig", newline="") as source:
        records = csv.reader(source, strict=True)
        header = next(row for row in records if row[:2] == ["Date", "Hour"])
        station_column = header.index("Tuen Mun")
        for row in records:
            if not row or all(not cell.strip() for cell in row):
                continue
            location = f"{path.name}:{records.line_num}"
            if len(row) != len(header):
                date_hour_issues.append((location, "Unexpected column count", row))
                continue
            date_text, hour_text = (cell.strip() for cell in row[:2])
            if hour_text == "Daily Max":
                continue

            if date_text:
                last_date = current_date
                current_date = None
                try:
                    if not re.fullmatch(r"[0-9]{4}-[0-9]{2}-[0-9]{2}", date_text):
                        raise ValueError
                    current_date = date.fromisoformat(date_text)
                    if (current_date.year, current_date.month) != (month.year, month.month):
                        raise ValueError
                except ValueError:
                    current_date = None
                    date_hour_issues.append((location, "Invalid date or wrong file month", date_text))
                    continue
                if last_date is not None and current_date != last_date + timedelta(days=1):
                    date_hour_issues.append((location, "Unexpected date-block order", date_text))

            if current_date is None or not re.fullmatch(r"0[1-9]|[1-9]|1[0-9]|2[0-4]", hour_text):
                date_hour_issues.append((location, "Missing date context or invalid hour", hour_text))
                continue
            hour = int(hour_text)
            if date_text and hour != 1 or not date_text and (
                previous_hour is None or previous_hour == 24 or hour != previous_hour + 1
            ):
                date_hour_issues.append((location, "Unexpected hour within date block", hour_text))
            previous_hour = hour

            raw_value = row[station_column].strip()
            substituted = raw_value.endswith("*")
            value_text = raw_value.removesuffix("*")
            if raw_value in {"", "-"}:
                aqhi = float("nan")
            elif re.fullmatch(r"[1-9]|10[+]?", value_text):
                aqhi = 11 if value_text == "10+" else int(value_text)
            else:
                invalid_aqhi.append((location, raw_value))
                aqhi = float("nan")

            timestamp = datetime.combine(current_date, datetime.min.time()) + timedelta(hours=hour)
            rows.append((timestamp, aqhi, substituted, raw_value, location))


In [2]:
source_data = pd.DataFrame(
    rows, columns=["timestamp", "aqhi", "is_substituted", "raw_aqhi", "source"]
)
source_data["timestamp"] = pd.to_datetime(source_data["timestamp"]).dt.tz_localize("Asia/Hong_Kong")
source_data["aqhi"] = source_data["aqhi"].astype("float64")
source_data = source_data.sort_values("timestamp", kind="stable").reset_index(drop=True)

expected = pd.date_range(
    "2025-04-01 01:00", "2026-07-01 00:00", freq="h", tz="Asia/Hong_Kong"
)
observed = pd.DatetimeIndex(source_data["timestamp"])
duplicates = source_data.loc[observed.duplicated(keep=False)]
missing_hours = expected.difference(observed)
unexpected_hours = observed.difference(expected)
cleaned = source_data[["timestamp", "aqhi", "is_substituted"]]
quality = pd.Series({
    "Total hourly records": len(cleaned),
    "Duplicate timestamps": duplicates["timestamp"].nunique(),
    "Rows involved in duplicates": len(duplicates),
    "Missing hours": len(missing_hours),
    "Hours outside the expected grid": len(unexpected_hours),
    "Missing AQHI (NaN)": int(cleaned["aqhi"].isna().sum()),
    "Original AQHI blanks": int(source_data["raw_aqhi"].eq("").sum()),
    "Original AQHI dashes": int(source_data["raw_aqhi"].eq("-").sum()),
    "Substituted readings": int(cleaned["is_substituted"].sum()),
    "Date/hour format or sequence issues": len(date_hour_issues),
    "Unrecognized AQHI tokens": len(invalid_aqhi),
    "10+ readings encoded as 11": int(cleaned["aqhi"].eq(11).sum()),
})
substituted = quality["Substituted readings"]
all_pct = 100 * substituted / len(cleaned)
available_pct = 100 * substituted / cleaned["aqhi"].count()

print(quality.to_string())
print(f"Substituted: {all_pct:.2f}% of all rows; {available_pct:.2f}% of nonmissing readings")
for label, details in [
    ("Duplicate rows and original source lines", duplicates),
    ("Missing hours", missing_hours),
    ("Unexpected hours", unexpected_hours),
    ("Date/hour issues", date_hour_issues),
    ("Unrecognized AQHI", invalid_aqhi),
]:
    if len(details):
        print(label, details, sep="\n")

print(cleaned.head(10).to_string(index=False))
print(cleaned.dtypes.to_string())


Total hourly records                   10944
Duplicate timestamps                       0
Rows involved in duplicates                0
Missing hours                              0
Hours outside the expected grid            0
Missing AQHI (NaN)                        18
Original AQHI blanks                      12
Original AQHI dashes                       6
Substituted readings                    1083
Date/hour format or sequence issues        0
Unrecognized AQHI tokens                   0
10+ readings encoded as 11                15
Substituted: 9.90% of all rows; 9.91% of nonmissing readings
                timestamp  aqhi  is_substituted
2025-04-01 01:00:00+08:00   2.0           False
2025-04-01 02:00:00+08:00   2.0           False
2025-04-01 03:00:00+08:00   2.0           False
2025-04-01 04:00:00+08:00   2.0           False
2025-04-01 05:00:00+08:00   2.0           False
2025-04-01 06:00:00+08:00   2.0           False
2025-04-01 07:00:00+08:00   2.0           False
2025-04-01 08:0

## Export and handoff
AQHI is an ordinal scale. `10+` is encoded as 11 to preserve ordering and can be recovered by reversing that mapping. Blank cells and literal `-` are represented as NaN and exported as empty cells. The export retains all valid time rows, including any duplicates, for review. Missing timestamps are reported without adding rows.

In [3]:
output_dir.mkdir(parents=True, exist_ok=True)
csv_path = output_dir / "tuen_mun_aqhi.csv"
# Report source issues first and prevent an export that silently drops invalid timestamps.
if date_hour_issues or invalid_aqhi:
    raise ValueError("Review the source issues printed above before exporting.")

cleaned.to_csv(csv_path, index=False, date_format="%Y-%m-%dT%H:%M:%S%z")
round_trip = pd.read_csv(csv_path)
round_trip["timestamp"] = pd.to_datetime(round_trip["timestamp"], utc=True).dt.tz_convert("Asia/Hong_Kong")
pd.testing.assert_frame_equal(round_trip, cleaned)

report = [
    "# Tuen Mun AQHI handoff",
    "",
    "Source: 15 Hong Kong EPD monthly English CSVs, April 2025 through June 2026.",
    "Reproduce with [01_data.ipynb](../../01_data.ipynb). Modeling input: [tuen_mun_aqhi.csv](tuen_mun_aqhi.csv).",
    "",
    "| Field | Type | Definition |",
    "|---|---|---|",
    "| timestamp | Timezone-aware datetime | Hong Kong hour endpoint (UTC+08:00). Source hour 24 is midnight the following day. |",
    "| aqhi | float64 | AQHI level. 11 encodes the official 10+ category. An empty CSV field parses as NaN. |",
    "| is_substituted | bool | True when EPD marked the reading with an asterisk, indicating a substitute station reading. |",
    "",
    "## Quality checks",
    "",
]
report.extend(f"- {label}: {count:,}" for label, count in quality.items())
report.extend([
    f"- Substituted share: {all_pct:.2f}% of all hourly rows and {available_pct:.2f}% of nonmissing AQHI.",
    f"- Timestamp coverage: {cleaned['timestamp'].iloc[0]} to {cleaned['timestamp'].iloc[-1]} inclusive.",
    "- The CSV contains hourly rows only. Original Daily Max rows remain in the source files.",
])
if len(missing_hours):
    report.extend(["", "Missing timestamps:", *[f"- {stamp.isoformat()}" for stamp in missing_hours]])
if len(duplicates):
    report.extend(["", "Duplicate rows (retained):", "```", duplicates.to_string(index=False), "```"])
(output_dir / "tuen_mun_aqhi_quality.md").write_text("\n".join(report) + "\n", encoding="utf-8")
print(f"Verified CSV round trip: {csv_path}")


Verified CSV round trip: data\processed\tuen_mun_aqhi.csv


## 來源、欄位及轉換規則

每行表示屯門站一個小時結束時刻的 AQHI。時間採香港時間 UTC+08:00。原始日期的 hour 01 對應同日 01:00，hour 24 對應次日 00:00。每月及訓練／測試統計使用原始報告日，即 timestamp 減一小時後的日期，讓月末 hour 24 留在來源月份。

| 欄位 | 格式及型別 | 意思及缺失值 |
|---|---|---|
| timestamp | ISO 8601，例如 2025-04-01T01:00:00+0800，讀入為帶時區 datetime | 小時結束時間，按時間升序 |
| aqhi | float64，1 至 10 及編碼 11 | 空白及 - 轉 NaN，CSV 寫成空欄，讀入不可填成 0 |
| is_substituted | bool，True / False | 星號讀數為 True，星號移除後的 AQHI 保留 |

環保署以 1–10 及 10+ 報告 AQHI，10+ 屬最高的 Serious 等級。11 是本項目的順序編碼，保留 10+ 類別資訊，不能視為環保署發布的精確 AQHI 數值。原始 CSV 說明星號表示站點／設備維修等暫停期間採用最相近站點的 AQHI，檔案沒有提供替代站名稱。空白表示該時段無 AQHI。- 的具體官方定義尚未在已查閱說明中找到，本項目把它作缺失並保留來源位置以便核對。

待 Damon 確認 10+ → 11 編碼及 hour 24 的次日午夜表示方式。現有交接 CSV 採此規則，尚未取得 Damon 的明確確認。本次 PR 提出確認事項，之後修正會另列受影響紀錄。

官方說明於 2026-11-19 查閱。規格 PDF 的日期範例為 16/10/2018，本批原件實際採 YYYY-MM-DD，解析及抽查以原件為準。

- [官方資料集](https://data.gov.hk/en-data/dataset/hk-epd-airteam-past-record-of-air-quality-health-index-en)
- [官方欄位規格](https://www.aqhi.gov.hk/psi/dd/hk_aqhien_en.pdf)
- [官方 AQHI 等級](https://www.aqhi.gov.hk/en/what-is-aqhi/about-aqhi.html)

source_manifest.csv 記錄每月下載網址、2026-11-17 下載日期及目前原件 SHA-256。下載日期根據專案實際下載紀錄，並非檔案修改日期或環保署發布日期。先前的 tuen_mun_aqhi_hourly.csv 是中間交接檔，建模輸入使用 tuen_mun_aqhi.csv。

In [4]:
import hashlib
import json
import platform
from datetime import timezone

manifest_rows, raw_counts = [], []
for month in months:
    path = raw_dir / f"{month.strftime('%Y%m')}_Eng.csv"
    with path.open(encoding="utf-8-sig", newline="") as stream:
        reader = csv.reader(stream, strict=True)
        next(row for row in reader if row[:2] == ["Date", "Hour"])
        body = [row for row in reader if row and any(row)]
    hourly_rows = [row for row in body if row[1] != "Daily Max"]
    raw_counts.append({
        "month": str(month), "raw_table_rows": len(body),
        "raw_hourly_rows": len(hourly_rows),
        "daily_max_rows_excluded": sum(row[1] == "Daily Max" for row in body),
        "blank_dates_filled": sum(not row[0] for row in hourly_rows),
    })
    manifest_rows.append({
        "month": str(month), "filename": path.name, "download_date": "2026-11-17",
        "download_date_basis": "project download session record",
        "source_url": f"https://www.aqhi.gov.hk/common/epd/ddata/html/history/{month.year}/{path.name}",
        "encoding": "UTF-8 with BOM", "bytes": path.stat().st_size,
        "sha256": hashlib.sha256(path.read_bytes()).hexdigest(),
    })
manifest = pd.DataFrame(manifest_rows)
before_after = pd.DataFrame(raw_counts)
manifest.to_csv(output_dir / "source_manifest.csv", index=False)
print(manifest[["month", "filename", "download_date", "bytes"]].to_string(index=False))
print(before_after.to_string(index=False))
print(f"Raw table rows: {before_after.raw_table_rows.sum():,}")
print(f"Daily maxima excluded: {before_after.daily_max_rows_excluded.sum():,}")
print(f"Hourly rows retained: {len(cleaned):,}")
print(f"Blank dates carried forward within source file: {before_after.blank_dates_filled.sum():,}")
assert before_after.raw_hourly_rows.sum() == len(cleaned)


  month       filename download_date  bytes
2025-04 202504_Eng.csv    2026-11-17  31803
2025-05 202505_Eng.csv    2026-11-17  32614
2025-06 202506_Eng.csv    2026-11-17  31680
2025-07 202507_Eng.csv    2026-11-17  32808
2025-08 202508_Eng.csv    2026-11-17  32647
2025-09 202509_Eng.csv    2026-11-17  31476
2025-10 202510_Eng.csv    2026-11-17  32631
2025-11 202511_Eng.csv    2026-11-17  31618
2025-12 202512_Eng.csv    2026-11-17  32688
2026-01 202601_Eng.csv    2026-11-17  32801
2026-02 202602_Eng.csv    2026-11-17  29589
2026-03 202603_Eng.csv    2026-11-17  32602
2026-04 202604_Eng.csv    2026-11-17  31734
2026-05 202605_Eng.csv    2026-11-17  32712
2026-06 202606_Eng.csv    2026-11-17  32382
  month  raw_table_rows  raw_hourly_rows  daily_max_rows_excluded  blank_dates_filled
2025-04             750              720                       30                 690
2025-05             775              744                       31                 713
2025-06             750              7

## 每月品質及訓練／測試期可用讀數

缺少時段是預期時間戳完全沒有紀錄。缺失讀數是已有該時間戳但 AQHI 為 NaN。duplicate_records 計同時間戳首筆以外的額外行，duplicate_timestamps 計有重複的時間戳，conflicting_timestamps 計 AQHI 或替代標記互相矛盾的時間戳。重複行保留於 cleaned，原始位置和明細列在下方。替代百分比的分母為組內所有小時行，AQHI ≥ 7 的有效讀數包括替代讀數。

分組依原始報告日，明確列出午夜邊界。最終 6 小時後目標及可建模樣本數由 Damon 確認。

In [5]:
def summarize_quality(frame, start, end):
    grid = pd.date_range(start, end, freq="h", tz="Asia/Hong_Kong")
    actual = pd.DatetimeIndex(frame["timestamp"])
    absent = grid.difference(actual)
    repeated = frame.loc[frame["timestamp"].duplicated(keep=False)]
    conflicts = sum(
        len(group[["aqhi", "is_substituted"]].drop_duplicates()) > 1
        for _, group in repeated.groupby("timestamp")
    )
    return {
        "expected_hours": len(grid), "actual_records": len(frame),
        "actual_unique_hours": actual.nunique(), "missing_hours": len(absent),
        "missing_timestamps": "|".join(t.isoformat() for t in absent),
        "missing_readings": int(frame["aqhi"].isna().sum()),
        "blank_readings": int(frame["raw_aqhi"].eq("").sum()),
        "dash_readings": int(frame["raw_aqhi"].eq("-").sum()),
        "duplicate_records": int(actual.duplicated().sum()),
        "duplicate_timestamps": repeated["timestamp"].nunique(),
        "conflicting_timestamps": conflicts,
        "substituted_readings": int(frame["is_substituted"].sum()),
        "substituted_pct": round(100 * frame["is_substituted"].mean(), 4),
        "valid_readings": int(frame["aqhi"].notna().sum()),
        "valid_station_readings": int((frame.aqhi.notna() & ~frame.is_substituted).sum()),
        "high_aqhi_readings": int(frame["aqhi"].ge(7).sum()),
    }

source_data["reporting_date"] = (source_data.timestamp - pd.Timedelta(hours=1)).dt.date
monthly_rows = []
for month in months:
    start, finish = month.start_time.date(), (month + 1).start_time.date()
    frame = source_data.loc[
        (source_data.reporting_date >= start) & (source_data.reporting_date < finish)
    ]
    monthly_rows.append({
        "month": str(month),
        **summarize_quality(frame, str(start) + " 01:00", str(finish) + " 00:00"),
    })
monthly_quality = pd.DataFrame(monthly_rows)
monthly_quality.to_csv(output_dir / "data_quality_summary.csv", index=False)
print(monthly_quality.drop(columns="missing_timestamps").to_string(index=False))

split_rows = []
for name, start, finish in [
    ("train", "2025-04-01", "2026-04-01"), ("test", "2026-04-01", "2026-07-01"),
]:
    frame = source_data.loc[
        (source_data.reporting_date >= date.fromisoformat(start))
        & (source_data.reporting_date < date.fromisoformat(finish))
    ]
    split_rows.append({
        "split": name, "reporting_date_start": start,
        "reporting_date_end": str(date.fromisoformat(finish) - timedelta(days=1)),
        "first_timestamp": frame.timestamp.min().isoformat(),
        "last_timestamp": frame.timestamp.max().isoformat(),
        **summarize_quality(frame, start + " 01:00", finish + " 00:00"),
    })
split_quality = pd.DataFrame(split_rows)
split_quality.to_csv(output_dir / "train_test_quality_summary.csv", index=False)
assert split_quality.actual_records.sum() == monthly_quality.actual_records.sum() == len(cleaned)
print("\nTraining and test availability")
print(split_quality.drop(columns="missing_timestamps").to_string(index=False))
print("\nDuplicate rows retained with source locations")
print(duplicates.to_string(index=False))
print("\nMissing timestamp detail")
print(missing_hours)


  month  expected_hours  actual_records  actual_unique_hours  missing_hours  missing_readings  blank_readings  dash_readings  duplicate_records  duplicate_timestamps  conflicting_timestamps  substituted_readings  substituted_pct  valid_readings  valid_station_readings  high_aqhi_readings
2025-04             720             720                  720              0                 0               0              0                  0                     0                       0                    49           6.8056             720                     671                  85
2025-05             744             744                  744              0                 0               0              0                  0                     0                       0                    27           3.6290             744                     717                   5
2025-06             720             720                  720              0                 0               0              0         

## 對照原件的抽查及完整核驗

重新打開原始 CSV，用 DictReader 獨立讀入 Date、Hour 及 Tuen Mun。把 timestamp 減去原始 hour 應得到原始日期的午夜，以此檢查日期補齊及跨日／跨月轉換。對全部小時行核對 AQHI、缺失狀態及替代標記。每月抽查首日 01／12／24 小時及月末 24 小時，另查全部空白／-、每月首個星號及首個 10+。source_spot_checks.csv 保存原始檔名、實際行號、日期是否空白及結果。

In [6]:
independent = []
for month in months:
    path = raw_dir / f"{month.strftime('%Y%m')}_Eng.csv"
    with path.open(encoding="utf-8-sig", newline="") as stream:
        for header_line_number, line in enumerate(stream, start=1):
            if line.startswith("Date,Hour,"):
                break
        reader = csv.DictReader(stream, fieldnames=header)
        carried_date = None
        for row in reader:
            if not row["Hour"] or row["Hour"] == "Daily Max":
                continue
            if row["Date"]:
                carried_date = datetime.fromisoformat(row["Date"])
            hour = int(row["Hour"])
            endpoint = pd.Timestamp(carried_date + timedelta(hours=hour), tz="Asia/Hong_Kong")
            token = row["Tuen Mun"]
            number = token.rstrip("*")
            numeric = float("nan") if number in ("", "-") else float(11 if number == "10+" else number)
            independent.append({
                "timestamp": endpoint, "aqhi": numeric, "is_substituted": "*" in token,
                "source_file": path.name, "source_line": header_line_number + reader.line_num,
                "raw_date": row["Date"], "carried_date": str(carried_date.date()),
                "source_hour": hour, "raw_aqhi": token, "month": str(month),
            })
independent = pd.DataFrame(independent).sort_values("timestamp", kind="stable").reset_index(drop=True)
pd.testing.assert_frame_equal(
    independent[["timestamp", "aqhi", "is_substituted"]], cleaned, check_dtype=False
)
for row in independent.itertuples():
    recovered = row.timestamp - pd.Timedelta(hours=row.source_hour)
    assert recovered.hour == 0 and str(recovered.date()) == row.carried_date

samples = {}
for month, group in independent.groupby("month", sort=True):
    candidates = [(group.iloc[i], reason) for i, reason in [
        (0, "month_start_hour01"), (11, "first_day_hour12"),
        (23, "first_day_hour24"), (-1, "month_end_hour24"),
    ]]
    candidates.extend((row, "missing_reading") for _, row in group.loc[group.aqhi.isna()].iterrows())
    for mask, reason in [
        (group.is_substituted, "substituted_reading"),
        (group.raw_aqhi.str.contains("10+", regex=False), "10plus_reading"),
    ]:
        selected = group.loc[mask]
        if len(selected):
            candidates.append((selected.iloc[0], reason))
    for row, reason in candidates:
        key = (row.source_file, int(row.source_line))
        if key not in samples:
            samples[key] = {**row.to_dict(), "reason": reason, "result": "PASS"}
        else:
            samples[key]["reason"] += "|" + reason
spot_checks = pd.DataFrame(samples.values()).sort_values("timestamp").reset_index(drop=True)
spot_checks.to_csv(output_dir / "source_spot_checks.csv", index=False)
print(f"Independent full source comparison: {len(independent):,} / {len(cleaned):,} rows PASS")
print(f"Documented source spot checks: {len(spot_checks)}")
print(spot_checks[[
    "source_file", "source_line", "raw_date", "source_hour", "timestamp",
    "raw_aqhi", "aqhi", "is_substituted", "reason", "result",
]].to_string(index=False))


Independent full source comparison: 10,944 / 10,944 rows PASS
Documented source spot checks: 93
   source_file  source_line   raw_date  source_hour                 timestamp raw_aqhi  aqhi  is_substituted                                 reason result
202504_Eng.csv            9 2025-04-01            1 2025-04-01 01:00:00+08:00        2   2.0           False                     month_start_hour01   PASS
202504_Eng.csv           20                      12 2025-04-01 12:00:00+08:00       2*   2.0            True   first_day_hour12|substituted_reading   PASS
202504_Eng.csv           32                      24 2025-04-02 00:00:00+08:00        3   3.0           False                       first_day_hour24   PASS
202504_Eng.csv          349                      16 2025-04-14 16:00:00+08:00      10+  11.0           False                         10plus_reading   PASS
202504_Eng.csv          757                      24 2025-05-01 00:00:00+08:00        4   4.0           False                     

## 重跑及輸出差異紀錄

從專案根目錄執行 Notebook 全部儲存格，或執行 python scripts/run_data_notebook.py。需要 Python 3、pandas 及 Asia/Hong_Kong 時區資料。下方記錄版本、來源與產物雜湊，把本次輸出和執行前已交接的 CSV 比較，結果存至 data_preparation_run.json。

In [7]:
hashes_after = {
    row["filename"]: hashlib.sha256((raw_dir / row["filename"]).read_bytes()).hexdigest()
    for row in manifest_rows
}
assert hashes_after == raw_hashes_before
current_hash = hashlib.sha256(csv_path.read_bytes()).hexdigest()
if original_handoff is not None:
    pd.testing.assert_frame_equal(original_handoff, pd.read_csv(csv_path))
    assert current_hash == original_handoff_hash
run_record = {
    "executed_at_utc": datetime.now(timezone.utc).isoformat(),
    "python": platform.python_version(), "pandas": pd.__version__,
    "input_files": len(manifest_rows), "input_sha256": hashes_after,
    "raw_table_rows": int(before_after.raw_table_rows.sum()),
    "daily_max_rows_excluded": int(before_after.daily_max_rows_excluded.sum()),
    "dates_filled": int(before_after.blank_dates_filled.sum()),
    "output_records": len(cleaned), "full_source_checks_passed": len(independent),
    "documented_spot_checks_passed": len(spot_checks),
    "handoff_csv_existed_before_run": original_handoff is not None,
    "handoff_csv_changed": current_hash != original_handoff_hash if original_handoff_hash else None,
    "duplicates_dropped": 0, "missing_rows_inserted": 0,
    "raw_files_unchanged": hashes_after == raw_hashes_before,
    "damon_conversion_confirmation": "pending",
}
artifacts = [
    csv_path, output_dir / "data_quality_summary.csv",
    output_dir / "train_test_quality_summary.csv", output_dir / "source_manifest.csv",
    output_dir / "source_spot_checks.csv",
]
run_record["output_sha256"] = {
    str(path): hashlib.sha256(path.read_bytes()).hexdigest() for path in artifacts
}
(output_dir / "data_preparation_run.json").write_text(
    json.dumps(run_record, indent=2, ensure_ascii=False) + "\n", encoding="utf-8"
)
print(json.dumps(run_record, indent=2, ensure_ascii=False))


{
  "executed_at_utc": "2026-11-19T11:37:51.537098+00:00",
  "python": "3.12.14",
  "pandas": "3.0.1",
  "input_files": 15,
  "input_sha256": {
    "202504_Eng.csv": "4f43a6416f2649a41c3120a8c7acebc4b5fc5b6b4e72df9a0b621da0fbe7a58a",
    "202505_Eng.csv": "862dc3f64824e572a1059c34747c80b137b1b57ece04f6ec8d8840ac5f82b6bb",
    "202506_Eng.csv": "e3c5f0ce26f53b580d849bae72f82097a59ff9f58490eb5169d2887c44eb25fa",
    "202507_Eng.csv": "d9ac11be12a78c11d4302e4bfc1142cc9f8f4a7327aeac533420c878541172ae",
    "202508_Eng.csv": "8d46b965f639775c9e79c3b876d85ac99194a8c46bc8e6f1194f9f059c31f67a",
    "202509_Eng.csv": "7f0892535f9bfd6b8a1d51b79ee79699da8a399e0ec6714bfbf72a896b43b7cf",
    "202510_Eng.csv": "0ffac57239547eb7f86e8ea4067ad23e13f9089a65be4fed9db1fb506aa83e9d",
    "202511_Eng.csv": "768ca7a2db5e212197c16167c63214070f361ac6b0b2ddb15125e70159c56e84",
    "202512_Eng.csv": "3d87343eacf89eb66f7a2610abe9de8b20d397ea3556def4f0b0cdb62867f623",
    "202601_Eng.csv": "b7f389e548b24ca671c2c62

In [8]:
quality_path = output_dir / "tuen_mun_aqhi_quality.md"
quality_text = quality_path.read_text(encoding="utf-8")
quality_text += """
## Sources and interpretation

Each row is the Hong Kong hour endpoint. Source hour 24 is midnight on the next day. Monthly quality and train/test tables group by the source reporting date (timestamp minus one hour). CSV timestamps contain an explicit +0800 offset.

EPD reports 1–10 and the 10+ category. The project encodes 10+ as 11 to retain its ordinal category. This is a project encoding and does not establish an exact continuous value. Asterisks indicate AQHI adopted from a similar station during station/equipment suspension. Missing values are exported as empty cells and read as NaN. The literal dash has no definition in the consulted specification and is treated as missing.

- [EPD dataset](https://data.gov.hk/en-data/dataset/hk-epd-airteam-past-record-of-air-quality-health-index-en)
- [EPD field specification](https://www.aqhi.gov.hk/psi/dd/hk_aqhien_en.pdf)
- [EPD AQHI scale](https://www.aqhi.gov.hk/en/what-is-aqhi/about-aqhi.html)

Official documents were consulted on 2026-11-19. Download dates in source_manifest.csv come from the project download session on 2026-11-17.

## Additional handoff files

- [Monthly quality summary](data_quality_summary.csv)
- [Training and test availability](train_test_quality_summary.csv)
- [Source manifest and SHA-256](source_manifest.csv)
- [Source spot checks with line numbers](source_spot_checks.csv)
- [Execution record and output SHA-256](data_preparation_run.json)
- [Report and presentation data text](../../docs/data_preparation.md)

In the monthly summary, duplicate_records counts extra rows beyond the first per timestamp. duplicate_timestamps counts affected timestamp keys. conflicting_timestamps counts keys with different AQHI or substitution flags. missing_hours counts absent timestamps. missing_readings counts existing rows with NaN. missing_timestamps lists missing endpoints with ISO 8601 timestamps. substituted_pct uses all hourly rows in the group as its denominator.

## Confirmation for Damon

Please confirm the existing 10+ to 11 mapping and the next-day-midnight timestamp for source hour 24 before finalizing model inputs. These rules are documented but explicit confirmation remains pending. Split counts describe cleaned readings. Final modeling sample counts depend on the six-hour-ahead target, feature availability, missing-data choices and treatment of targets across split boundaries.
"""
quality_path.write_text(quality_text, encoding="utf-8")
print("Updated field definitions, quality report, source references and pending confirmation.")


Updated field definitions, quality report, source references and pending confirmation.
